In [181]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [182]:
scaled_train = pd.read_csv('../data/processed/scaled_train.csv')
scaled_val = pd.read_csv('../data/processed/scaled_val.csv')

In [183]:
X, y = scaled_train.drop(columns=["result"]), scaled_train["result"]

In [184]:
X_val = scaled_val.drop(columns=["result"])
y_val = scaled_val["result"]

In [185]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score


model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss",
    early_stopping_rounds=20
)

model.fit(
    X,
    y,
    eval_set=[(X_val, y_val)],
    verbose=True
)

# Predict
y_pred = model.predict(X)
y_val_pred = model.predict(X_val)

probs = model.predict_proba(X)

print(probs.shape)
print(probs[:5])

# Evaluate
print("Accuracy:", accuracy_score(y, y_pred))
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))

[0]	validation_0-mlogloss:1.08059
[1]	validation_0-mlogloss:1.07621
[2]	validation_0-mlogloss:1.07279
[3]	validation_0-mlogloss:1.06937
[4]	validation_0-mlogloss:1.06399
[5]	validation_0-mlogloss:1.05910
[6]	validation_0-mlogloss:1.05579
[7]	validation_0-mlogloss:1.05273
[8]	validation_0-mlogloss:1.04967
[9]	validation_0-mlogloss:1.04747
[10]	validation_0-mlogloss:1.04376
[11]	validation_0-mlogloss:1.04286
[12]	validation_0-mlogloss:1.03844
[13]	validation_0-mlogloss:1.03633
[14]	validation_0-mlogloss:1.03626
[15]	validation_0-mlogloss:1.03397
[16]	validation_0-mlogloss:1.03240
[17]	validation_0-mlogloss:1.03127
[18]	validation_0-mlogloss:1.03019
[19]	validation_0-mlogloss:1.02918
[20]	validation_0-mlogloss:1.02749
[21]	validation_0-mlogloss:1.02749
[22]	validation_0-mlogloss:1.02672
[23]	validation_0-mlogloss:1.02492
[24]	validation_0-mlogloss:1.02422
[25]	validation_0-mlogloss:1.02281
[26]	validation_0-mlogloss:1.02178
[27]	validation_0-mlogloss:1.02136
[28]	validation_0-mlogloss:1.0

In [186]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.36      0.53       184
           1       0.66      0.91      0.76       299
           2       0.72      0.72      0.72       223

    accuracy                           0.71       706
   macro avg       0.78      0.66      0.67       706
weighted avg       0.75      0.71      0.69       706



In [187]:
import pandas as pd

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance)

                     feature  importance
19              ranking_diff    0.091084
10     away_relative_shots_2    0.051561
12              away_ranking    0.049040
3       home_shots_against_1    0.047008
16     home_stadium_temp_avg    0.046404
18   away_stadium_wind_speed    0.045898
11              home_ranking    0.045372
20   stadium_temperature_avg    0.043806
17     away_stadium_temp_avg    0.043393
14           away_wind_speed    0.041650
0   away_stadium_distance_km    0.041327
1                 home_fix_1    0.040689
8                 away_fix_2    0.040178
4       home_shots_against_2    0.039650
13           home_wind_speed    0.039264
5      home_relative_shots_1    0.039036
21      home_temperature_avg    0.038774
15        stadium_wind_speed    0.037967
22      away_temperature_avg    0.037023
6      home_relative_shots_2    0.036994
7                 away_fix_1    0.036980
2                 home_fix_2    0.033981
9      away_relative_shots_1    0.032922


In [188]:
import joblib
joblib.dump(model, '../models/xgboost.pkl')

['../models/xgboost.pkl']

# Inference